# Web Scraping Data Processing: The Vibecoder's Workflow
Welcome! In this notebook, we'll focus on the often-messy reality of web scraping: **Data Cleaning (Transform)** and **Exporting (Load)**. You already know how to fetch the raw HTML, but knowing how to reliably extract and structure that data is what separates beginners from pros.

A crucial part of the **Vibecoder's workflow** is defining a clear data schema *before* writing the cleaning logic. We need to envision exactly what our final data should look like (e.g., a dictionary with specific keys and strict data types) so we can design a robust pipeline to tame the chaos of the web.

***

In [1]:
# 1. Setup & Mock Data
# Let's simulate some messy data we just scraped from e-commerce product cards.
# Notice the hidden newlines, inconsistent spacing, strings masquerading as numbers,
# and the missing price tag entirely!

scraped_mock_data = [
    {
        "title": "\n  Sony Alpha a7 III Camera \n",
        "price_string": "$1998.00  ",
        "stock": "In Stock"
    },
    {
        "title": "   Keychron K2 Mechanical Keyboard\n",
        "price_string": " $79.99\n",
        "stock": "Out of Stock"
    },
    {
        "title": "\nGeneric USB-C Cable  ",
        # UH OH! Notice there is no "price_string" key here at all. 
        # This simulates a product card where the price element was missing on the webpage.
        "stock": "In Stock"
    },
    {
        "title": "Logitech MX Master 3 Mouse\n",
        "price_string": "$99.99",
        "stock": " \n In Stock \n"
    }
]

print("Raw Scraped Data Example:", scraped_mock_data[0])

Raw Scraped Data Example: {'title': '\n  Sony Alpha a7 III Camera \n', 'price_string': '$1998.00  ', 'stock': 'In Stock'}


## Phase 1: Cleaning and Type Conversion (Transform)

Web data is unstructured text by default. A classic programmatic trap is leaving prices as strings (like `"$19.99"`). If you ever want to find the average price, sort by price, or run analytics later on, you *must* convert them to numbers (like `19.99` into a Python `float`). 

We'll use Python's built-in `.strip()` to surgically remove hidden spaces and newlines (`\n`). 

**Handling the Unexpected:** Web layouts change, and elements disappear. If your Python script blindly assumes a price always exists, it *will* crash with a `KeyError` or a `ValueError` eventually. We use a combination of `.get()` and defensive `try/except` logic to gracefully handle missing data. 

***

In [2]:
# We will construct our clean list representing our ideal schema
cleaned_data = []

# Loop through our chaotic mock data
for item in scraped_mock_data:
    
    # 1. Initialize a clean dictionary for this specific item
    # This represents a strict target schema we want to enforce.
    clean_item = {}
    
    # 2. Clean the Strings (Title & Stock)
    # Use .get() to pull the raw text, defaulting to an empty string if it's missing.
    # We immediately chain .strip() to destroy leading/trailing whitespace and \n characters.
    clean_item["title"] = item.get("title", "").strip()
    clean_item["stock_status"] = item.get("stock", "").strip()
    
    # 3. Clean the Price (The Tricky Part!)
    # Again, dictionary.get() is vital. If the key is totally missing (like our USB cable),
    # it safely returns None instead of crashing the program with a KeyError.
    raw_price = item.get("price_string")
    
    if raw_price is not None: 
        try:
            # Step A: Remove spaces and newlines
            stripped_price = raw_price.strip()
            
            # Step B: Remove the dollar sign
            number_string = stripped_price.replace("$", "")

            
            # Step C: Convert the string representation of a number to an actual float!
            clean_item["price"] = float(number_string)
            
        except ValueError:
            # DEFENSIVE PROGRAMMING:
            # If float() fails (e.g., the website text unexpectedly said "Contact for Price"), 
            # we catch the ValueError so the spider doesn't break down entirely!
            clean_item["price"] = None 
            print(f"Warning: Could not parse price into float for '{clean_item['title']}'")
            
    else:
        # If the price tag was missing completely (like our USB-C cable), 
        # we assign a fallback value (None) to maintain our schema structure cleanly.
        clean_item["price"] = None
        print(f"Warning: Price missing completely for '{clean_item['title']}'")
        
    # Finally, append the finalized, strictly-typed item to our master list
    cleaned_data.append(clean_item)

print("\n--- Final Cleaned Data ---")
for data in cleaned_data:
    print(data)


--- Final Cleaned Data ---
{'title': 'Sony Alpha a7 III Camera', 'stock_status': 'In Stock', 'price': 1998.0}
{'title': 'Keychron K2 Mechanical Keyboard', 'stock_status': 'Out of Stock', 'price': 79.99}
{'title': 'Generic USB-C Cable', 'stock_status': 'In Stock', 'price': None}
{'title': 'Logitech MX Master 3 Mouse', 'stock_status': 'In Stock', 'price': 99.99}


## Phase 2: Exporting the Data (Load)

Now that our data is sanitized and strictly conforms to a predictable schema (a uniform list of dictionaries), moving it to a permanent storage medium is a breeze.

We will export our data in two standard formats:
1. **JSON (`.json`)**: The backbone of web-dev. Excellent for passing data between web APIs, storing in NoSQL databases like MongoDB, or injecting into a Django backend.
2. **CSV (`.csv`)**: A data analysis classic. Excellent for viewing the data easily in Excel, or importing it into pandas / SQL. We'll use the `pandas` library to convert our dictionaries to a CSV table in literally two lines of code!

***

In [3]:
import json
import pandas as pd # Make sure you have pandas installed! (pip install pandas)

# --- Built-in Method: Exporting as JSON ---

# We open a new file named "cleaned_products.json" in "w" (write) mode.
# Setting encoding="utf-8" ensures weird characters (like emojis on the webpage) don't corrupt the file.
with open("cleaned_products.json", "w", encoding="utf-8") as json_file:
    # json.dump pushes our structured Python list directly into the file.
    # indent=4 is entirely optional, but it formats the file to be highly readable to humans.
    json.dump(cleaned_data, json_file, indent=4)
    
print("✅ Successfully exported data to cleaned_products.json!")

# --- The Analyst's Workflow: Exporting via Pandas to CSV ---

# Pandas is arguably the kingslayer library for data manipulation.
# We load our list of dictionaries into a DataFrame (a 2D spreadsheet layout in memory).
df = pd.DataFrame(cleaned_data)

print("\n Pandas DataFrame Preview:")
print("-" * 30)
print(df.head())
print("-" * 30)

# Export the DataFrame straight to our filesystem as a CSV file.
# setting index=False correctly tells pandas NOT to write its internal row numbers (0, 1, 2) to the file.
df.to_csv("cleaned_products.csv", index=False, encoding="utf-8")

print("\n✅ Successfully exported data to cleaned_products.csv!")

✅ Successfully exported data to cleaned_products.json!

 Pandas DataFrame Preview:
------------------------------
                             title  stock_status    price
0         Sony Alpha a7 III Camera      In Stock  1998.00
1  Keychron K2 Mechanical Keyboard  Out of Stock    79.99
2              Generic USB-C Cable      In Stock      NaN
3       Logitech MX Master 3 Mouse      In Stock    99.99
------------------------------

✅ Successfully exported data to cleaned_products.csv!
